# TaxonRouter test

Tests `TaxonRouter.route()` across species from different kingdoms to verify kingdom detection and API list selection.

In [ ]:
from scripts.apis_pipe.gbif import GBIFAPI
from scripts.utils.router import TaxonRouter

gbif = GBIFAPI()
router = TaxonRouter(gbif_client=gbif)

## GBIF Backbone Kingdoms

Where can we find all the kingdom names? The GBIF Backbone Taxonomy (`d7dddbf4-2cf0-4f39-9b2a-bb099caae36c`) has this information. This link shows the first 9 rows with RANK == KINGDOM (the last row is PHYLUM): [https://api.gbif.org/v1/species?datasetKey=d7dddbf4-2cf0-4f39-9b2a-bb099caae36c&limit=10](https://api.gbif.org/v1/species?datasetKey=d7dddbf4-2cf0-4f39-9b2a-bb099caae36c&limit=10)

The backbone contains **8 accepted kingdoms** (plus "incertae sedis" which is Latin for "of uncertain placement", or unknown).

In [2]:
import requests

GBIF_BACKBONE = "d7dddbf4-2cf0-4f39-9b2a-bb099caae36c"
resp = requests.get(
    "https://api.gbif.org/v1/species",
    params={"datasetKey": GBIF_BACKBONE, "limit": 20},
)
kingdoms = [r for r in resp.json()["results"] if r.get("rank") == "KINGDOM"]

for k in kingdoms:
    name = k["canonicalName"]
    status = k["taxonomicStatus"]
    print(f"  {name:<20} ({status:<10})")

  incertae sedis       (DOUBTFUL  )
  Animalia             (ACCEPTED  )
  Archaea              (ACCEPTED  )
  Bacteria             (ACCEPTED  )
  Chromista            (ACCEPTED  )
  Fungi                (ACCEPTED  )
  Plantae              (ACCEPTED  )
  Protozoa             (ACCEPTED  )
  Viruses              (ACCEPTED  )


## Helper

In [3]:
def test_route(name: str):
    kingdom = router._get_kingdom(name)
    apis = router.route(name)
    print(f"{name}")
    print(f"  Kingdom : {kingdom}")
    print(f"  APIs    : {apis}")
    print()

## Fungi — *Amanita muscaria*

In [4]:
test_route("Amanita muscaria")

Amanita muscaria
  Kingdom : Fungi
  APIs    : ['gbif', 'col', 'genbank', 'index_fungorum', 'mushroomobs', 'symbiota_mycoportal', 'symbiota_lichen']



## Plantae — *Quercus robur*

In [5]:
test_route("Quercus robur")

Quercus robur
  Kingdom : Plantae
  APIs    : ['gbif', 'col', 'genbank', 'tropicos', 'symbiota_bryophyte', 'symbiota_cch2', 'symbiota_sernec', 'symbiota_nansh', 'symbiota_swbiodiversity', 'symbiota_macroalgae', 'symbiota_pterido', 'symbiota_neherbaria', 'symbiota_midatlantic']



## Animalia — *Panthera leo*

In [6]:
test_route("Panthera leo")

Panthera leo
  Kingdom : Animalia
  APIs    : ['gbif', 'col', 'genbank']



## Unknown / gibberish name -- should return an empty list

In [7]:
test_route("Xyzzyus notarealus")

Xyzzyus notarealus
  Kingdom : Unknown
  APIs    : []

